In [63]:
## Python v3.10.13

from __future__ import annotations
import numpy as np
import xlwings as xw
import pickle
import pandas as pd
import os
import sys
# sys.path.append('/home/wlh3/wagner')
from openai import AsyncOpenAI

from pydantic_ai import Agent
from openai import AsyncOpenAI
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic import BaseModel, Field, create_model
from typing import Dict, List, Any, Optional, Type, Literal
import asyncio
from enum import Enum
import json
import re
sys.path.append('/Users/williamharrigan/Desktop/test_wagner')
import creds

In [64]:
def collapse_rows(df):
    """Collapse multiple rows into one, combining different values into lists."""
    def combine_values(series):
        # Flatten any existing lists in the series first
        flat_vals = []
        for val in series.dropna():
            if isinstance(val, list):
                flat_vals.extend(val)
            else:
                flat_vals.append(val)
        
        # Deduplicate case-insensitively, keeping the first seen version
        seen_lower = set()
        seen = []
        for v in flat_vals:
            v_lower = v.lower() if isinstance(v, str) else v
            if v_lower not in seen_lower:
                seen_lower.add(v_lower)
                seen.append(v)
        
        if len(seen) == 0:
            return None
        elif len(seen) == 1:
            return seen[0]
        else:
            return seen

    return df.groupby('species_key', dropna=False).agg(combine_values).reset_index()

def is_missing(val):
    # unwrap string-encoded dicts - replace nan with None before parsing
    if isinstance(val, str):
        try:
            cleaned = re.sub(r'\bnan\b', 'None', val)
            val = ast.literal_eval(cleaned)
        except (ValueError, SyntaxError):
            if val.strip() in ('', 'None', 'nan'):
                return True
            return False  

    if val is None or val == 'None' or val == 'nan':
        return True
    elif isinstance(val, float) and np.isnan(val):
        return True
    elif isinstance(val, list) and len(val) == 0:
        return True
    elif isinstance(val, dict):
        relevant_keys = ['exmin', 'min', 'max', 'exmax']
        relevant_vals = [val.get(k) for k in relevant_keys]
        def is_null(v):
            if v is None:
                return True
            if isinstance(v, pd.Series):
                v = v.iloc[0]
            if isinstance(v, float) and np.isnan(v):
                return True
            return False
        return all(is_null(v) for v in relevant_vals)
    return False

def dict_to_df(d: dict) -> pd.DataFrame:
    serialized = {}
    for k, v in d.items():
        if isinstance(v, Enum):
            serialized[k] = v.value
        elif isinstance(v, list):
            serialized[k] = [x.value if isinstance(x, Enum) else x for x in v]
        else:
            serialized[k] = v
    return pd.DataFrame([serialized])

def collapse_type_columns(df):
    # Identify all columns that have '_type_' in their names
    type_cols = [col for col in df.columns if '_type_' in col]

    # Group columns by their prefix (everything before '_type_' + last element)
    col_groups = {}
    for col in type_cols:
        prefix = '_'.join(col.split('_')[:-1])  # Get the prefix
        if prefix not in col_groups:
            col_groups[prefix] = []
        col_groups[prefix].append(col)

    # Create new collapsed columns
    for prefix, cols in col_groups.items():
        df[prefix] = df.apply(lambda row: [col.split('_')[-1].upper() for col in cols if row[col] == 1], axis=1)
        
    # Drop the original type columns
    df.drop(columns=type_cols, inplace=True)

    return df

def merge_hawaiian_name_columns(df):
    # Identify columns that contain 'hawaiian_name'
    hawaiian_cols = [col for col in df.columns if 'hawaiian_name' in col]

    # Ensure there are columns to merge
    if not hawaiian_cols:
        return df
    
    # Merge values into a list, ensuring all values are strings and filtering out empty values
    df['hawaiian_name'] = df[hawaiian_cols].apply(lambda row: [str(val) for val in row if pd.notna(val) and str(val).strip() != ''], axis=1)

    # Drop the original columns
    df.drop(columns=hawaiian_cols, inplace=True)

    return df

KNOWN_UNITS = {'mm', 'cm', 'm', 'km', 'g', 'kg', 'mg', 'ml', 'l', 'um', 'μm'}

def collapse_measurements(df):
    col_suffix_map = {}
    base_columns = {}

    for col in df.columns:
        parts = col.split('_')
        if len(parts) > 2 and parts[-2] == 'extreme' and parts[-1] in ('min', 'max'):
            base_name = '_'.join(parts[:-2])
            suffix_key = 'ex' + parts[-1]
        elif len(parts) > 1 and parts[-1] in ('exmin', 'min', 'max', 'exmax', 'unit'):
            base_name = '_'.join(parts[:-1])
            suffix_key = parts[-1]
        else:
            continue

        col_suffix_map[col] = (base_name, suffix_key)
        if base_name not in base_columns:
            base_columns[base_name] = []
        base_columns[base_name].append(col)

    non_measurement_cols = [col for col in df.columns if col not in col_suffix_map]
    result_df = df[non_measurement_cols].copy()

    collapsed_data = {}

    for base_name, cols in base_columns.items():
        collapsed_data[base_name] = []

        for _, row in df.iterrows():
            measurement_dict = {'exmin': None, 'min': None, 'max': None, 'exmax': None, 'unit': None}
            for col in cols:
                _, suffix_key = col_suffix_map[col]
                val = row[col]
                if suffix_key == 'unit':
                    # Take unit directly from the _unit column value
                    measurement_dict['unit'] = val if pd.notna(val) else None
                else:
                    measurement_dict[suffix_key] = val if pd.notna(val) else None
            collapsed_data[base_name].append(measurement_dict)

    collapsed_df = pd.DataFrame(collapsed_data)
    result_df = pd.concat([result_df, collapsed_df], axis=1)

    return result_df

def parse_chrom_values(val):
    if not isinstance(val, list):
        return np.nan
    
    numbers = []
    for item in val:
        item = str(item).strip()
        # Handle ranges like "14–16" or "14-16"
        match = re.match(r'(\d+)[–\-](\d+)', item)
        if match:
            numbers.append((int(match.group(1)) + int(match.group(2))) / 2)
        elif item.isdigit():
            numbers.append(int(item))
    
    return int(np.mean(numbers)) if numbers else np.nan

def parse_measurement_string(s):
    if not isinstance(s, str):
        return s
    
    # Extract unit
    unit_match = re.search(r"'unit':\s*'(\w+)'", s)
    unit = unit_match.group(1) if unit_match else None
    
    return {
        'exmin': np.nan,
        'min': np.nan,
        'max': np.nan,
        'exmax': np.nan,
        'unit': unit
    }

In [67]:
## Import manually extracted data for comparison
# manual_extracted_data_df = pd.read_csv(f"/Users/williamharrigan/Desktop/UH/Year_4/Wagner/pkl_files/clean_full_wagner.csv")
manual_extracted_data_df = pd.read_csv(f"/Users/williamharrigan/Desktop/UH/Year_4/Wagner/gold_standard_data/clean_full_wagner_annotated.csv")
manual_extracted_data_df['species_key'] = (
    manual_extracted_data_df['family'].astype(str) + '_' +
    manual_extracted_data_df['genus'].astype(str) + '_' +
    manual_extracted_data_df['species'].astype(str)
)

# manual_extracted_data_df.drop('Unnamed: 0.1', axis=1, inplace=True)
# manual_extracted_data_df.drop('Unnamed: 0', axis=1, inplace=True)
manual_extracted_data_df['species_key'] = manual_extracted_data_df['species_key'].str.lower()
manual_extracted_data_df['female_calyx_length'] = manual_extracted_data_df['female_calyx_length'].apply(parse_measurement_string)
# manual_extracted_data_df = collapse_rows(manual_extracted_data_df)

manual_extracted_data_df.head()

,family,genus,species,common_name,wagner_pg_number,description,infraspecific_epithet,stem_hair_type,phyllotaxy_type,breeding_type,inflorescence_type,ray_color,floret_color,spathe_color,perianth_outer_color,perianth_inner_color,perianth_color,labellum_color,corolla_type,staminate_corolla_type,pistillate_corolla_type,corolla_color,fruit_type,ploidy,chromosome_number,average_chromosome_number,origin,status,life_form_type,leaf_type,leaf_margin_type,leaf_shape_type,juvenile_leaf_type,juvenile_leaf_margin_type,juvenile_leaf_shape_type,leaflets_leaf_type,leaflets_leaf_margin_type,leaflets_leaf_shape_type,leaf_hair_type,leaf_hair_upper_type,leaf_hair_lower_type,juvenile_leaf_hair_type,island_type,hawaiian_name,stem_height,leaf_length,leaf_width,juvenile_leaf_length,juvenile_leaf_width,leaflets_leaf_length,leaflets_leaf_width,petioles,staminate_inflorescence_length,staminate_inflorescence_width,pistillate_inflorescence_length,pistillate_inflorescence_width,inflorescence_flower_length,inflorescence_flower_width,flower_length,flower_width,rachis_length,rachis_diameter,head_length,head_diameter,bur_length,tepal_length,staminate_tepal_length,pistillate_tepal_length,ray_length,ray_width,florets_length,involucre_length,involucre_width,staminate_involucre_length,pistilate_involucre_length,bract_length,bract_width,bract_lower_length,bract_outer_length,bracteoles_length,bracteoles_width,pedicel_length,pedicel_width,pistillate_pedicel_length,hypanthium_length,hypanthium_width,peduncle_length,peduncle_width,spathe_length,spathe_width,spadix_length,perianth_length,perianth_width,perianth_outer_length,perianth_outer_width,perianth_inner_length,perianth_inner_width,perianth_tube_length,perianth_lobes_length,perianth_lobes_width,staminate_perianth_tube_length,pistillate_perianth_tube_length,pappus_length,umbellet_length,labellum_length,labellum_width,calyx_length,calyx_width,calyx_teeth_length,calyx_teeth_width,calyx_lobes_length,calyx_lobes_width,upper_calyx_length,lower_calyx_length,inner_calyx_lobes_length,inner_calyx_lobes_width,outer_calyx_lobes_length,outer_calyx_lobes_width,calyx_tube_length,calyx_tube_width,male_calyx_length,male_calyx_width,male_calyx_lobes_length,male_calyx_lobes_width,female_calyx_length,female_calyx_width,female_calyx_lobes_length,female_calyx_lobes_width,male_calyx_lobes_length_inner,male_calyx_lobes_length_outer,male_calyx_lobes_width_outer,male_calyx_tube_length,female_calyx_lobes_length_inner,female_calyx_lobes_length_outer,female_calyx_lobes_width_inner,female_calyx_lobes_width_outer,inner_calyx_length,outer_calyx_length,corolla_length,corolla_width,corolla_tube_length,corolla_tube_width,corolla_lobes_length,corolla_lobes_width,upper_corolla_length,lower_corolla_length,upper_corolla_lobes_length,lower_corolla_lobes_length,corolla_lip_length,staminate_corolla_length,pistillate_corolla_length,staminate_corolla_tube_length,staminate_corolla_tube_width,pistillate_corolla_tube_length,pistillate_corolla_tube_width,female_corolla_lobes_length,female_corolla_lobes_width,male_corolla_lobes_length,male_corolla_lobes_width,fruit_length,fruit_width,fruit_diameter,seeds_perfruit,seed_length,seed_width,seed_diameter,pistillate_peduncle_length,pistillate_peduncle_width,staminate_pedicel_length,staminate_peduncle_length,staminate_peduncle_width,species_key
0,Acanthaceae,Asystasia,gangetica,"Chinese_violet, Asystasia",pg 168-169,Dicots,"micrantha, gangetica",STRIGOSE,DECUSSATE,MONOECIOUS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ZYGOMORPHIC,NaN,NaN,Pale blue to purple or white or tinged yellow,CAPSULE,2n,"26, 28, 44, 48, 50, 52",41.3333333,NATURALIZED,NaN,['PERENNIAL_HERB'],['SIMPLE'],['NO TEETH'],NaN,NaN,NaN,NaN,NaN,NaN,NaN,['STRIGOSE'],NaN,NaN,NaN,"['MAUI', 'MOLOKAI', 'OAHU', 'KAUAI']",NaN,"{'exmin': nan, 'min': 0.5, 'max': 2.0, 'exmax'...","{'exmin': nan, 'min': 2.5, 'max': 13.0, 'exmax...","{'exmin': nan, 'min': 1.5, 'max': 4.0, 'exmax'...",NaN,NaN,NaN,NaN,"{'exmin': nan, 'min': 0.5, 'max': 4.5, 'exmax'...","{'exmin': nan, 'min': nan, 'max

In [20]:
# ----------------------------
# OpenAI client + model
# ----------------------------
openai_client = AsyncOpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

model = OpenAIChatModel(
    "gpt-5.4",
    provider=OpenAIProvider(openai_client=openai_client),
)



In [21]:
import re

def parse_text_inputs(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    
    sections = re.split(r'-{3,}INPUT FOR (.+?)-{3,}', content)
    
    text_inputs = {}
    # sections[0] is empty, then alternates: name, content, name, content...
    for i in range(1, len(sections), 2):
        key = sections[i].strip().replace(' ', '_')
        value = sections[i + 1].strip() if i + 1 < len(sections) else ''
        text_inputs[key] = value
    
    return text_inputs

text_inputs = parse_text_inputs('/Users/williamharrigan/Desktop/test_wagner/wagner_species.txt')

In [22]:
# ----------------------------
# Enums
# ----------------------------
class Description(str, Enum):
    DICOTS = "Dicots"
    MONOCOTS = "Monocots"
    CONIFERS = "Conifers"
    FERNS = "Ferns and fern allies"

class LifeFormType(str, Enum):
    ANNUAL_HERB = "ANNUAL_HERB"
    BIENNIAL_HERB = "BIENNIAL_HERB"
    PERENNIAL_HERB = "PERENNIAL_HERB"
    EPIPHYTE = "EPIPHYTE"
    VINE = "VINE"
    SHRUB = "SHRUB"
    TREE = "TREE"
    SHRUB_TREE = "SHRUB/TREE"
    GRASS = "GRASS"
    SEDGE = "SEDGE"


class StemHairType(str, Enum):
    DENDRITIC = "DENDRITIC"
    GLABROUS = "GLABROUS"
    HIRSUTE = "HIRSUTE"
    HISPID = "HISPID"
    LEPIDOTE = "LEPIDOTE"
    PILOSE = "PILOSE"
    PUBERULENT = "PUBERULENT"
    STRIGOSE = "STRIGOSE"
    STELLATE = "STELLATE"
    TOMENTOSE = "TOMENTOSE"
    VILLOUS = "VILLOUS"
    GLAUCOUS = "GLAUCOUS"

class LeafType(str, Enum):
    SIMPLE = "SIMPLE"
    COMPOUND = "COMPOUND"


class LeafShapeType(str, Enum):
    ACEROSE = "ACEROSE"
    AWL_SHAPED = "AWL_SHAPED"
    GLADIATE = "GLADIATE"
    HASTATE = "HASTATE"
    CORDATE = "CORDATE"
    DELTOID = "DELTOID"
    # LANCEOLATE = "LANCEOLATE"
    # LINEAR = "LINEAR"
    ELLIPTIC = "ELLIPTIC"
    ENSIFORM = "ENSIFORM"
    LYRATE = "LYRATE"
    OBCORDATE = "OBCORDATE"
    FALCATE = "FALCATE"
    FLABELLATE = "FLABELLATE"
    OBDELTOID = "OBDELTOID"
    OBELLIPTIC = "OBELLIPTIC"
    OBLANCEOLATE = "OBLANCEOLATE"
    OBLONG = "OBLONG"
    PERFOLIATE = "PERFOLIATE"
    QUADRATE = "QUADRATE"
    OBOVATE = "OBOVATE"
    ORBICULAR = "ORBICULAR"
    RENIFORM = "RENIFORM"
    RHOMBIC = "RHOMBIC"
    OVAL = "OVAL"
    OVATE = "OVATE"
    ROTUND = "ROTUND"
    SAGITTATE = "SAGITTATE"
    PANDURATE = "PANDURATE"
    PELTATE = "PELTATE"
    SPATULATE = "SPATULATE"
    SUBULATE = "SUBULATE"
    CUNEATE = "CUNEATE"


class LeafMarginType(str, Enum):
    TEETH = "TEETH"
    LOBED = "LOBED"
    ENTIRE = "ENTIRE"
    NOTEETH = "NO TEETH"


class PhyllotaxyType(str, Enum):
    ALTERNATE = "ALTERNATE"
    OPPOSITE = "OPPOSITE"
    WHORLED = "WHORLED"
    DECUSSATE = "DECUSSATE"
    DISTICHOUS = "DISTICHOUS"
    # EQUITANT = "EQUITANT"
    # TERNATE = "TERNATE"
    # CAULINE = "CAULINE"

class LeafHairType(str, Enum):
    DENDRITIC = "DENDRITIC"
    GLABROUS = "GLABROUS"
    HIRSUTE = "HIRSUTE"
    HISPID = "HISPID"
    LEPIDOTE = "LEPIDOTE"
    PILOSE = "PILOSE"
    PUBERULENT = "PUBERULENT"
    STRIGOSE = "STRIGOSE"
    STELLATE = "STELLATE"
    TOMENTOSE = "TOMENTOSE"
    VILLOUS = "VILLOUS"
    GLAUCOUS = "GLAUCOUS"

class InflorescenceType(str, Enum):
    AXILLARY = "AXILLARY"
    CATKIN = "CATKIN"
    CYME = "CYME"
    CLUSTERS = "CLUSTERS"
    DICHASIUM = "DICHASIUM"
    FASCICLE = "FASCICLE"
    GLOMERATE = "GLOMERATE"
    HEAD = "HEAD"
    PANICLE = "PANICLE"
    RACEME = "RACEME"
    SPATHE_SPADIX = "SPATHE_SPADIX"
    THYRSE = "THYRSE"
    UMBEL = "UMBEL"
    VERTISCILLATE = "VERTISCILLATE"
    SOLITARY = "SOLITARY"
    SPIKE = "SPIKE"
    SPADIX = "SPADIX"


class CorollaType(str, Enum):
    TUBULAR = "TUBULAR"
    CAMPANULATE = "CAMPANULATE"
    FUNNELFORM = "FUNNELFORM"
    ROTATE = "ROTATE"
    SALVERFORM = "SALVERFORM"
    BILABIATE = "BILABIATE"
    ZYGOMORPHIC = "ZYGOMORPHIC"
    URCEOLATE = "URCEOLATE"
    CUP = "CUP"
    OBOVATE = "OBOVATE"
    ELIPTIC = "ELLIPTIC"
    OBLONG = "OBLONG"
    ORBICULAR = "ORBICULAR"
    


class BreedingType(str, Enum):
    MONOECIOUS = "MONOECIOUS"
    DIOECIOUS = "DIOECIOUS"
    POLYGAMOUS = "POLYGAMOUS"
    ANDROMONOECIOUS = "ANDROMONOECIOUS"
    GYNODIOECIOUS = "GYNODIOECIOUS"
    CLEISTOGAMOUS = "CLEISTOGAMOUS"


class Location(str, Enum):
    HAWAII = "HAWAII"
    MAUI = "MAUI"
    KAHOOLAWE = "KAHOOLAWE"
    MOLOKAI = "MOLOKAI"
    LANAI = "LANAI"
    OAHU = "OAHU"
    KAUAI = "KAUAI"
    NIIHAU = "NIIHAU"
    ALL_ISLANDS = "ALL ISLANDS"


class OriginType(str, Enum):
    INDIGENOUS = "INDIGENOUS"
    ENDEMIC = "ENDEMIC"
    PC = "PALEOTROPICAL COSMOPOLITAN - Originating from the paleotropics"
    NATURALIZED = "NATURALIZED"

class StatusType(str, Enum):
    NATURALIZED = "NATURALIZED"
    ENDEMIC = "ENDEMIC"
    RARE = "RARE"
    SECURE = "SECURE"
    VULNERABLE = "VULNERABLE"


class FruitType(str, Enum):
    ACHENE = "ACHENE"
    BERRY = "BERRY"
    CAPSULE = "CAPSULE"
    DRUPE = "DRUPE"
    LEGUME = "LEGUME"
    NUT = "NUT"
    MERICARP = "MERICARP"

class Measurements(BaseModel):
    "Extract units as described in passage. No conversions."
    min: Optional[float] = None
    max: Optional[float] = None
    extreme_min: Optional[float] = None
    extreme_max: Optional[float] = None
    unit: Optional[str] = None
    
class Dimensions(BaseModel):
    length: Optional[Measurements] = Field(None, description="Extract length")
    width: Optional[Measurements] = Field(None, description="Extract width")


In [49]:
### Data Fields

class CoreFieldsModel(BaseModel):
    "Core fields that should be extracted for every plant. Family, genus and species names described by labels in the text."
    family: str = Field(..., description="Plant family name")
    genus: str = Field(..., description="Plant genus name")
    species: str = Field(..., description="Plant species epithet")
    infraspecific_epithet: Optional[str] = Field(None, description="Infraspecific taxon name following the species name")
    common_name: Optional[str] = Field(None, description="Common or vernacular name of the plant")
    hawaiian_name: Optional[List[str]] = Field(None, description="Hawaiian name of the plant, including diacritical marks")
    wagner_pg_number: Optional[str] = Field(None, description="Reference page number in Wagner's Manual of Flowering Plants of Hawaii")
    description: Optional[Description] = Field(None, description="Major plant group classification (infer from taxonomy)")
    
class OuterFlowerMorphologyModel(BaseModel):
    corolla_type: Optional[List[CorollaType]] = Field(None, description="The shape/description of petals (e.g., oblong, obovate, lanceolate)")
    corolla_color: Optional[str] = Field(None, description="Color of petals (corolla)")
    perianth_color: Optional[str] = Field(None, description="Color of perianth (undifferentiated tepals)")
    perianth_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of collective outer envelope (must include petals, calyx, sepals) of flower.")
    corolla_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of flower petals.")
    calyx_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of flower sepals.")
    labellum_color: Optional[str] = Field(None, description="Color of labellum/lip (modified orchid petal)")
    
    labellum_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of labellum/lip.")
    
    calyx_teeth_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the lobes of the calyx margin.")
    calyx_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of calyx lobes in flowers with fused sepals.")
    
    upper_calyx_length: Optional[Measurements] = Field(None, description="Length of upper flower sepals specifically.")
    lower_calyx_length: Optional[Measurements] = Field(None, description="Length of lower flower sepals specifically.")
    inner_calyx_length: Optional[Measurements] = Field(None, description="Length of inner flower sepals specifically.")
    outer_calyx_length: Optional[Measurements] = Field(None, description="Length of outer flower sepals specifically.")
    
    inner_calyx_lobes_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of inner calyx lobes when flowers sepals are joined together.")
    outer_calyx_lobes_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of outer calyx lobes when flowers sepals are joined together.")
    
    male_calyx_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the male (staminate) flower calyx specifically.") 
    female_calyx_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the female (pistillate) flower calyx specifically.")
    
    male_calyx_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the lobes of male (staminate) flower calyx.")
    male_calyx_inner_lobe_length: Optional[Measurements] = Field(None, description="Length of the inner lobes of male (staminate) flower calyx.")
    male_calyx_outer_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the outer lobes of male (staminate) flower calyx.")
    
    female_calyx_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the lobes of female (pistillate) flower calyx.")
    female_calyx_inner_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the inner lobes of female (pistillate) flower calyx.")
    female_calyx_outer_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the outer lobes of female (pistillate) flower calyx.")
    
    calyx_tube_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the tubular, fused base of sepals (calyx tube).")
    male_calyx_tube_length: Optional[Measurements] = Field(None, description="Length of the tubular, fused base of sepals (calyx tube) on male (staminate) flowers.")
    
    corolla_tube_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the tubular, fused structure of the petals (corolla tube).")
    corolla_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the free parts of fused petals (corolla lobe).")
    corolla_lip_length: Optional[Measurements] = Field(None, description="Length of the corolla lip or labellum which is a specialized lobe of flowers with fused petals.")
    
    upper_corolla_length: Optional[Measurements] = Field(None, description="Length of upper flower petals (corolla) specifically.")
    lower_corolla_length: Optional[Measurements] = Field(None, description="Length of lower flower petals (corolla) specifically.")
    
    upper_corolla_lobes_length: Optional[Measurements] = Field(None, description="Length of upper lobes or lip of the fused petals (corolla).")
    lower_corolla_lobes_length: Optional[Measurements] = Field(None, description="Length of lower lobes or lip of the fused petals (corolla).")
    
    staminate_corolla_length: Optional[Measurements] = Field(None, description="Length of staminate flower petals (corolla) specifically.")
    pistillate_corolla_length: Optional[Measurements] = Field(None, description="Length of pistillate flower petals (corolla) specifically.")
    
    staminate_corolla_tube_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the tubular, fused structure of the petals (corolla tube) of male (staminate) flowers.")
    pistillate_corolla_tube_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the tubular, fused structure of the petals (corolla tube) of female (pistillate) flowers.")

    male_corolla_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the free parts of fused petals (corolla lobe) of male (staminate) flowers.")
    female_corolla_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the free parts of fused petals (corolla lobe) of female (pistillate) flowers.")
    
    perianth_inner_color: Optional[str] = Field(None, description="Color of inner perianth specifically.")
    perianth_outer_color: Optional[str] = Field(None, description="Color of outer perianth specifically.")
    perianth_outer_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the outer collective of the outer outer envelope of flower (must include corolla, calyx, sepals)")
    perianth_inner_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the inner collective of the outer outer envelope of flower (must include corolla, calyx, sepals)")
    perianth_tube_length: Optional[Measurements] = Field(None, description="Length of the tubular structure formed by the perianth when tepals are fused.")
    perianth_lobes_dimensions: Optional[Dimensions] = Field(None, description="Length of the lobes formed by the fusion of perianth segments.")
    
    staminate_perianth_tube_length: Optional[Measurements] = Field(None, description="Length of the tubular structure formed by the perianth when tepals are fused on staminate/male flowers.")
    pistillate_perianth_tube_length: Optional[Measurements] = Field(None, description="Length of the tubular structure formed by the perianth when tepals are fused on pistillate/female flowers.")
    
    tepal_length: Optional[Measurements] = Field(None, description="Length of tepals when petals and sepals are not differentiated.")
    staminate_tepal_length: Optional[Measurements] = Field(None, description="Length of tepals of male/staminate plants")
    pistillate_tepal_length: Optional[Measurements] = Field(None, description="Length of tepals of female/pistillate plants")
    flower_dimensions: Optional[Dimensions] = Field(None, description="Dimensions given for the overall flower.")


class FruitMorphologyModel(BaseModel):
    fruit_type: Optional[List[FruitType]] = Field(None, description="Type of  mature, ripened ovary of a flowering plant or fruit described.")
    fruit_length: Optional[Measurements] = Field(None, description="Fruit length")
    fruit_width: Optional[Measurements] = Field(None, description="Fruit width")
    fruit_diameter: Optional[Measurements] = Field(None, description="Fruit diameter")
    bur_length: Optional[Measurements] = Field(None, description="Bur (spiny fruit covering) length")
    seeds_perfruit: Optional[Measurements] = Field(None, description="Number of seeds per fruit")
    seed_length: Optional[Measurements] = Field(None, description="Seed length.")
    seed_width: Optional[Measurements] = Field(None, description="Seed width")
    seed_diameter: Optional[Measurements] = Field(None, description="Seed diameter")    
    
class InflorescenceSpecificMorphologyFieldsModel(BaseModel):
    head_length: Optional[Measurements] = Field(None, description="Capitulum length")
    head_diameter: Optional[Measurements] = Field(None, description="Capitulum diameter")
    pappus_length: Optional[Measurements] = Field(None, description="Pappus length.")
    ray_dimensions: Optional[Dimensions] = Field(None, description="Describes dimensions of rays or ray florets.")
    ray_color: Optional[str] = Field(None, description="Ray floret color")
    floret_color: Optional[str] = Field(None, description="Color of florets (must specify florets vs flower/petal color)")
    florets_length: Optional[Measurements] = Field(None, description="Length of small flowers that compose a flower head (capitulum)")
    spathe_color: Optional[str] = Field(None, description="Color of spathe")
    spathe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of spathe")
    spadix_length: Optional[Measurements] = Field(None, description="Length of spadix")

class BractInvolucreMorphologyFieldsModel(BaseModel):
    bract_dimensions: Optional[Dimensions] = Field(None, description="Bract dimensions.")
    bract_lower_length: Optional[Measurements] = Field(None, description="Lower/basal bract length")
    bract_outer_length: Optional[Measurements] = Field(None, description="Outer bract length")
    bracteoles_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of bracteoles")
    involucre_dimensions: Optional[Dimensions] = Field(None, description="Involucre dimensions")    
    staminate_involucre_length: Optional[Measurements] = Field(None, description="Length of involucre of staminate/male heads.")
    pistillate_involucre_length: Optional[Measurements] = Field(None, description="Length of involucre of pistillate/female heads.")
    
class InflorescenceMorphologyModel(BaseModel):
    inflorescence_type: Optional[List[InflorescenceType]] = Field(None, description="Type of inflorescence")
    inflorescence_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the inflorescence.")
    
    peduncle_dimensions: Optional[Dimensions] = Field(None, description="Main inflorescence (peduncle) stalk dimensions")
    pedicel_dimensions: Optional[Dimensions] = Field(None, description="Individual flower stalk (pedicel) in an inflorescence dimensions")
    
    rachis_length: Optional[Measurements] = Field(None, description="Main inflorescence stalk (rachis) length")
    rachis_diameter: Optional[Measurements] = Field(None, description="Main inflorescence stalk (rachis) diameter")
    
    hypanthium_dimensions: Optional[Dimensions] = Field(None, description="Floral cup (hypanthium) dimensions")
    umbellet_length: Optional[Measurements] = Field(None, description="Secondary umbel unit length")
    
    staminate_inflorescence_dimensions: Optional[Dimensions] = Field(None, description="Male inflorescence dimensions")
    pistillate_inflorescence_dimensions: Optional[Dimensions] = Field(None, description="Female inflorescence dimensions")

    staminate_peduncle_dimensions: Optional[Dimensions] = Field(None, description="Main inflorescence (peduncle) stalk of male/staminate plant dimensions")
    pistillate_peduncle_dimensions: Optional[Dimensions] = Field(None, description="Main inflorescence (peduncle) stalk of female/pistillate plant dimensions")
    
    staminate_pedicel_dimensions: Optional[Dimensions] = Field(None, description="Individual flower stalk (pedicel) in an inflorescence of male/staminate plant dimensions")
    pistillate_pedicel_dimensions: Optional[Dimensions] = Field(None, description="Individual flower stalk (pedicel)  in an inflorescence of female/pistillate plant dimensions")

class LeafMorphologyModel(BaseModel):
    leaf_type: Optional[List[LeafType]] = Field(None, description="Leaf structure type (simple or compound)")
    leaf_shape_type: Optional[List[LeafShapeType]] = Field(None, description="Shape of leaf blade (ovate, lanceolate, cordate, etc.)")
    leaf_margin_type: Optional[List[LeafMarginType]] = Field(None, description="Type of leaf edge (entire, serrate, lobed, etc.)")
    phyllotaxy_type: Optional[List[PhyllotaxyType]] = Field(None, description="The arrangement of leaves around the stem (words like pairs and leaflet information are important indicators).")

    leaf_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of leaves or individual leaf blade")
    petiole_length: Optional[Measurements] = Field(None, description="Length of leaf stalk (petiole).")

class LeafIndumentumModel(BaseModel):
    leaf_hair_type: Optional[List[LeafHairType]] = Field(None, description="Type of leaf surface indumentum.")
    upper_leaf_hair_type: Optional[List[LeafHairType]] = Field(None, description="Specific description of upper leaf surface indumentum.")
    lower_leaf_hair_type: Optional[List[LeafHairType]] = Field(None, description="Specific description of lower leaf surface indumentum.")

class LeafletMorphologyFieldsModel(BaseModel):
    leaflets_leaf_type: Optional[List[LeafType]] = Field(None, description="Leaflet leaf structure type in compound leaves")
    leaflets_leaf_shape_type: Optional[List[LeafShapeType]] = Field(None, description="Shape of individual leaflets")
    leaflets_leaf_margin_type: Optional[List[LeafMarginType]] = Field(None, description="Margin type of leaflets")
    leaflets_leaf_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of individual leaflets")
    
class JuvenileLeafFieldsModel(BaseModel):
    juvenile_leaf_type: Optional[List[LeafType]] = Field(None, description="Leaf structure type in juvenile plants")
    juvenile_leaf_shape_type: Optional[List[LeafShapeType]] = Field(None, description="Shape of juvenile leaves")
    juvenile_leaf_margin_type: Optional[List[LeafMarginType]] = Field(None, description="Margin type of juvenile leaves")
    juvenile_leaf_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of juvenile leaves")
    juvenile_leaf_hair_type: Optional[List[LeafHairType]] = Field(None, description="Indumentum type on juvenile leaves")

class StemMorphologyModel(BaseModel):
    stem_height: Optional[Measurements] = Field(None, description="Overall height of the plant or length of the main stem")
    stem_hair_type: Optional[List[StemHairType]] = Field(None, description="Describes the stem hair type of the main stem (NOT penduncle) (e.g., glabrous, hirsute, tomentose, etc.)")
    
class LifeFormModel(BaseModel):
    life_form_type: Optional[List[LifeFormType]] = Field(None, description="Growth habit or life form (annual herb, perennial herb, shrub, tree, vine, etc.)")
    breeding_type: Optional[List[BreedingType]] = Field(None, description="Plant reproductive class (infer from taxonomy)")

class ReproductiveMorphologyModel(BaseModel):
    # life_form_type: Optional[List[LifeFormType]] = Field(None, description="Growth habit or life form")
    # breeding_type: Optional[List[BreedingType]] = Field(None, description="Plant reproductive class (monoecious if male and female organs are on the same plant, dioecious if on different plants).")
    ploidy: Optional[list] = Field(None, description="Ploidy level for the specific species expressed as a function of n (e.g., 1n, 2n or 3n, etc..)")
    chromosome_number: Optional[list] = Field(None, description="Species specific chromosome count (integer number)")
    
class DistributionFieldsModel(BaseModel):
    island_type: Optional[List[Location]] = Field(None, description= "Hawaiian islands the plant is found.")
    origin: Optional[List[OriginType]] = Field(None, description="Origin abbreviated at the beginning of the passage (end/endemic, nat/naturalized, PC/paleotropical cosmopoltian.)")
    status: Optional[List[StatusType]] = Field(None, description="Conservation or rarity status")
    

In [50]:
# ----------------------------
# Flag models
# ----------------------------

def FlagField(desc: str):
    return Field(default_factory=Flag, description=desc)

class Flag(BaseModel):
    value: bool = Field(False, description="Set True if the passage clearly provides specific information for the given field.")
    description: str = Field("", description="Short justification or quote to justify the flag or 'not mentioned'.")
    
def to_bools(self) -> Dict[str, bool]:
    """Flatten to {flag_name: bool}."""
    return {k: getattr(self, k).value for k in self.model_fields}

def reasons(self) -> Dict[str, str]:
    """Flatten to {flag_name: description}."""
    return {k: getattr(self, k).description for k in self.model_fields}

In [51]:
class GroupFlags(BaseModel):
    has_core:             Flag = FlagField("Contains core taxonomic/identification metadata.")
    has_life_form: Flag = FlagField("Describes life form such as annual herb, shrub, tree, vine, etc.")
    has_outer_flower_morphology: Flag = FlagField("Describes outer flower morphology including petals, corolla, calyx, perianth, or sepal morphology.")
    has_fruit_morphology: Flag = FlagField("Describes fruit morphology such as fruit type, size, or color.")
    has_inflorescence_specific_morphology: Flag = FlagField("Describes or mentions head, pappus, spadix, spathe, ray, rays, or floret morphology.")
    has_bract_involucre_morphology: Flag = FlagField("Describes involucre, bract or bracteole morphology.")
    has_inflorescence_morphology: Flag = FlagField("Describes inflorescence morphology such as type of flower cluster arrangement (cyme, raceme, etc.) or dimensions of inflorescence parts.")
    has_leaf_morphology: Flag = FlagField("Describes leaf morphology such as leaf type, shape, margin, or phyllotaxy.")
    has_leaflet_morphology: Flag = FlagField("Specifically mentions leaflet morphology.")
    has_juvenile_leaf_morphology: Flag = FlagField("Specifically mentions juvenile leaf morphology.")
    has_stem_morphology: Flag = FlagField("Describes stem morphology such as plant height or hair type.")
    has_reproductive_morphology: Flag = FlagField("Describes reproductive morphology such as life form, breeding type, flower dimensions, or cytology.")
    has_distribution: Flag = FlagField("Describes plant distribution such as specific Hawaiian islands, origin status, or conservation status.")
    # has_bract: Flag = FlagField("Describes bract morphology such as bract dimensions or color.")
    has_leaf_indumentum: Flag = FlagField("Describes leaf indumentum such as hair type of leaf surface.")
    # has_specific_outer_flower_morphology: Flag = FlagField("Describes specific aspects of outer flower morphology such as calyx, perianth, or sepal lobes, tubes, or teeth.")
    # has_plant_group_classification: Flag = FlagField("Can infer plant group classification such as dicot, monocot, fern, or conifer.")

In [52]:
FLAG_TO_MODEL: Dict[str, tuple] = {
    "has_core":                              ("core",                             CoreFieldsModel),
    "has_life_form":                             ("life_form",                             LifeFormModel),
    "has_outer_flower_morphology":                              ("outer_flower_morphology",                             OuterFlowerMorphologyModel),
    "has_fruit_morphology":                             ("fruit_morphology",                             FruitMorphologyModel),
    "has_inflorescence_specific_morphology":                             ("Inflorescence_Specific",                             InflorescenceSpecificMorphologyFieldsModel),
    "has_bract_involucre_morphology":                             ("bract_involucre_morphology",                             BractInvolucreMorphologyFieldsModel),
    "has_inflorescence_morphology":                             ("inflorescence_morphology",                             InflorescenceMorphologyModel),
    "has_leaf_morphology":                             ("leaf_morphology",                             LeafMorphologyModel),
    "has_leaflet_morphology":                             ("leaflet_morphology",                             LeafletMorphologyFieldsModel),
    "has_juvenile_leaf_morphology":                             ("juvenile_leaf_morphology",                             JuvenileLeafFieldsModel),
    "has_stem_morphology":                             ("stem_morphology",                             StemMorphologyModel),
    "has_reproductive_morphology":                             ("reproductive_morphology",                             ReproductiveMorphologyModel),
    "has_distribution":                             ("distribution",                             DistributionFieldsModel),
    # "has_bract":                             ("bract",                             BractMorphologyFieldsModel),
    "has_leaf_indumentum":                             ("leaf_indumentum",                             LeafIndumentumModel),
    # "has_specific_outer_flower_morphology":                             ("specific_outer_flower_morphology",                             SpecificOuterFlowerMorphologyModel),
    # "has_plant_group_classification":                             ("plant_group_classification",                             PlantGroupClassificationModel),
    }

In [11]:
# class GroupFlags(BaseModel):
#     # has_core:             Flag = FlagField("Contains core taxonomic/identification metadata.")
#     # has_life_form: Flag = FlagField("Describes life form such as annual herb, shrub, tree, vine, etc.")
#     # has_outer_flower_morphology: Flag = FlagField("Describes outer flower morphology including petals, corolla, calyx, perianth, or sepal morphology.")
#     # has_fruit_morphology: Flag = FlagField("Describes fruit morphology such as fruit type, size, or color.")
#     has_inflorescence_specific_morphology: Flag = FlagField("Describes head, pappus, spadix, spathe, rays, or floret morphology.")
#     # has_involucre_morphology: Flag = FlagField("Describes involucre, bract or bracteole morphology.")
#     # has_inflorescence_morphology: Flag = FlagField("Describes inflorescence morphology such as type of flower cluster arrangement (cyme, raceme, etc.) or dimensions of inflorescence parts.")
#     # has_leaf_morphology: Flag = FlagField("Describes leaf morphology such as leaf type, shape, margin, or phyllotaxy.")
#     # has_leaflet_morphology: Flag = FlagField("Describes leaflet morphology such as leaflet type, shape, margin, or dimensions.")
#     # has_juvenile_leaf_morphology: Flag = FlagField("Describes juvenile leaf morphology such as leaf type, shape, margin, or dimensions.")
#     # has_stem_morphology: Flag = FlagField("Describes stem morphology such as plant height or hair type.")
#     # has_reproductive_morphology: Flag = FlagField("Describes reproductive morphology such as life form, breeding type, flower dimensions, or cytology.")
#     # has_distribution: Flag = FlagField("Describes plant distribution such as specific Hawaiian islands, origin status, or conservation status.")
#     # has_bract: Flag = FlagField("Describes bract morphology such as bract dimensions or color.")
#     # has_leaf_indumentum: Flag = FlagField("Describes leaf indumentum such as hair type of leaf surface.")
#     # has_specific_outer_flower_morphology: Flag = FlagField("Describes specific aspects of outer flower morphology such as calyx, perianth, or sepal lobes, tubes, or teeth.")
#     # has_plant_group_classification: Flag = FlagField("Can infer plant group classification such as dicot, monocot, fern, or conifer.")

# FLAG_TO_MODEL: Dict[str, tuple] = {
#     # "has_core":                              ("core",                             CoreFieldsModel),
#     # "has_life_form":                             ("life_form",                             LifeFormModel),
#     # "has_outer_flower_morphology":                              ("outer_flower_morphology",                             OuterFlowerMorphologyModel),
#     # "has_fruit_morphology":                             ("fruit_morphology",                             FruitMorphologyModel),
#     "has_inflorescence_specific_morphology":                             ("Inflorescence_Specific",                             InflorescenceSpecificMorphologyFieldsModel),
#     # "has_involucre_morphology":                             ("involucre_morphology",                             InvolucresMorphologyFieldsModel),
#     # "has_inflorescence_morphology":                             ("inflorescence_morphology",                             InflorescenceMorphologyModel),
#     # "has_leaf_morphology":                             ("leaf_morphology",                             LeafMorphologyModel),
#     # "has_leaflet_morphology":                             ("leaflet_morphology",                             LeafletMorphologyFieldsModel),
#     # "has_juvenile_leaf_morphology":                             ("juvenile_leaf_morphology",                             JuvenileLeafFieldsModel),
#     # "has_stem_morphology":                             ("stem_morphology",                             StemMorphologyModel),
#     # "has_reproductive_morphology":                             ("reproductive_morphology",                             ReproductiveMorphologyModel),
#     # "has_distribution":                             ("distribution",                             DistributionFieldsModel),
#     # "has_bract":                             ("bract",                             BractMorphologyFieldsModel),
#     # "has_leaf_indumentum":                             ("leaf_indumentum",                             LeafIndumentumModel),
#     # "has_specific_outer_flower_morphology":                             ("specific_outer_flower_morphology",                             SpecificOuterFlowerMorphologyModel),
#     # "has_plant_group_classification":                             ("plant_group_classification",                             PlantGroupClassificationModel),
#     }

In [53]:
# ----------------------------
# Generic agent runner
# ----------------------------
async def _run_agent(result_type: type, system_prompt: str, sample: str, model) -> Any:
    agent = Agent(
        model=model,
        output_type=result_type,
        system_prompt=system_prompt,
    )
    resp = await agent.run(sample)
    return resp.output


# ----------------------------
# Orchestrator
# ----------------------------
async def extract_plant(sample: str, model) -> Dict[str, Any]:
    # Phase 1: detect all flags in parallel
    flag_names = list(GroupFlags.model_fields)
    flag_results = await asyncio.gather(*[
        _run_agent(
            Flag,
            f"Botanical specialist. Does the passage contain: {GroupFlags.model_fields[name].description} "
            "Set value=True only if clearly supported. Quote the relevant text.",
            sample, model,
        )
        for name in flag_names
    ], return_exceptions=True)

    flag_values: Dict[str, Flag] = {}
    for name, result in zip(flag_names, flag_results):
        if isinstance(result, Exception):
            print(f"Flag error [{name}]: {result}")
            flag_values[name] = Flag()
        else:
            flag_values[name] = result
    flags = GroupFlags(**flag_values)

    # Phase 2: extract all flagged groups in parallel (deduplicate shared models)
    seen: set = set()
    active: list[tuple[str, type, str]] = [
        (group_name, model_cls, getattr(flags, flag_name).description)
        for flag_name, (group_name, model_cls) in FLAG_TO_MODEL.items()
        if getattr(flags, flag_name).value and group_name not in seen and not seen.add(group_name)
    ]
    active_groups: Dict[str, type] = {group_name: model_cls for group_name, model_cls, _ in active}

    group_results = await asyncio.gather(*[
        _run_agent(
            model_cls,
            f"You are a botantical expert. Extract the data fields provided, in the text passage. Return null for fields that are not clearly stated.",
            sample, model,
        )
        for group_name, model_cls, flag_desc in active
    ], return_exceptions=True)

    results = {
        group_name: data
        for (group_name, _, _), data in zip(active, group_results)
        if not isinstance(data, Exception)
    }

    # Phase 3: validate and self-correct
    results = await validate_and_correct(sample, model, results, active_groups)

    return results, flags

In [54]:
# Groups unlikely to hallucinate — skip validation to save tokens
# SKIP_VALIDATION = {"core", "distribution", "cytology", "habit"}
# SKIP_VALIDATION = {"core", "fruit_morphology", "reproductive_morphology"}
SKIP_VALIDATION = {"core"}
# ----------------------------
# Validation models
# ----------------------------
class FieldFlag(BaseModel):
    field: str = Field(..., description= "Field name")
    reason: str = Field(..., description= "Reasoning field was flagged (hallucination or constraint violation)")

# class GroupValidation(BaseModel):
#     is_valid: bool = Field(default= True, description="False if an expert botantist would not be able to infer the same values.")
#     flagged_fields: List[FieldFlag] = Field(default_factory=list)

# List-based so the LLM doesn't have to return a dict (avoids ValidationError)
class GroupValidationItem(BaseModel):
    group_name: str
    is_valid: bool = Field(default=True, description="False if the values are clearly hallucinated or violate model constraints.")
    flagged_fields: List[FieldFlag] = Field(default_factory=list)

class BatchValidation(BaseModel):
    groups: List[GroupValidationItem]


# ----------------------------
# Extract species-level text (trim family/genus preamble for correction calls)
# ----------------------------
def extract_species_text(text: str) -> str:
    """Return species-level section only; fall back to full text if not found."""
    import re
    match = re.search(r'\nSpecies:.*', text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()
    cutoff = int(len(text) * 0.4)
    return text[-cutoff:] if cutoff < len(text) else text


# ----------------------------
# Parallel group re-extractor (used by validate_and_correct for corrections)
# ----------------------------
async def parse_all_groups_parallel(
    sample: str,
    model,
    group_models: Dict,
    group_hints: Dict[str, str] = None,
) -> Dict:
    group_hints = group_hints or {}

    async def run_one(name, cls):
        hint = group_hints.get(name, "")
        prompt = f"Botanical specialist. Extract only {name} fields. Return null for unsupported fields."
        if hint:
            prompt += f"\n{hint}"
        return name, await _run_agent(cls, prompt, sample, model)

    pairs = await asyncio.gather(*[run_one(n, c) for n, c in group_models.items()], return_exceptions=True)
    return {n: d for item in pairs if not isinstance(item, Exception) for n, d in [item]}


# ----------------------------
# Batch validate all groups in ONE call
# ----------------------------
async def validate_all_batch(sample: str, model, results: Dict) -> Dict[str, GroupValidationItem]:
    to_validate = {n: r for n, r in results.items() if n not in SKIP_VALIDATION}
    if not to_validate:
        return {}

    val_prompt = (
        "You are a botanical data quality auditor. "
        "Given source text and extracted data for multiple field groups, "
        "flag any field that:\n"
        "- contradicts the source text (hallucination)\n"
        "- is not specific to the data field or species being extracted.\n"
        "Leave is_valid=True and flagged_fields=[] if all fields are supported by the source text."
    )

    groups_json = json.dumps(
        {name: json.loads(r.model_dump_json()) for name, r in to_validate.items()},
        indent=2,
    )

    agent = Agent(
        model=model,
        output_type=BatchValidation,
        system_prompt=val_prompt,
    )
    resp = await agent.run(f"SOURCE TEXT:\n{sample}\n\nEXTRACTED DATA:\n{groups_json}")

    returned = {item.group_name: item for item in resp.output.groups}

    # Groups the LLM didn't mention are implicitly valid
    return {
        name: returned.get(name, GroupValidationItem(group_name=name))
        for name in to_validate
    }


# ----------------------------
# Validate + self-correct loop
# ----------------------------
async def validate_and_correct(
    sample: str,
    model,
    results: Dict,
    active_groups: Dict,
    group_hints: Dict[str, str] = None,
    max_retries: int = 2,
) -> Dict:
    current = dict(results)
    group_hints = group_hints or {}
    correction_text = sample

    for attempt in range(max_retries):
        validations = await validate_all_batch(sample, model, current)
        flagged = {n: v for n, v in validations.items() if not v.is_valid}

        if not flagged:
            print(f"All groups valid" + (f" after {attempt} correction(s)." if attempt else "."))
            return current

        print(f"Attempt {attempt + 1}/{max_retries}: correcting {list(flagged.keys())}")
        for name, val in flagged.items():
            for f in val.flagged_fields:
                print(f"  {name}.{f.field}: {f.reason}")

        correction_hints = {}
        for name in flagged:
            issues = "; ".join(f"{f.field}: {f.reason}" for f in flagged[name].flagged_fields)
            base = group_hints.get(name, "")
            correction_hints[name] = f"{base}\nCorrection notes: {issues}".strip()

        flagged_models = {n: active_groups[n] for n in flagged if n in active_groups}
        corrections = await parse_all_groups_parallel(correction_text, model, flagged_models, group_hints=correction_hints)
        current.update(corrections)

    return current

In [55]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
def _meas(m):
    if m is None:
        return None
    return {'exmin': m.extreme_min, 'min': m.min, 'max': m.max, 'exmax': m.extreme_max, 'unit': m.unit}

def result_to_row(result):
    c   = result.get('core')
    st  = result.get('stem_morphology')
    lf  = result.get('leaf_morphology')
    ll  = result.get('leaflet_morphology')
    jl  = result.get('juvenile_leaf_morphology')
    lft = result.get('life_form')
    rp  = result.get('reproductive_morphology')
    ds  = result.get('distribution')
    inf = result.get('inflorescence_morphology')
    fl  = result.get('outer_flower_morphology')
    fr  = result.get('fruit_morphology')
    cap = result.get('Inflorescence_Specific')
    inv = result.get('bract_involucre_morphology')
    li  = result.get('leaf_indumentum')
    br  = result.get('bract_involucre_morphology')

    fam = c.family  if c else None
    gen = c.genus   if c else None
    sp  = c.species if c else None

    def enums(obj, attr):
        v = getattr(obj, attr, None) if obj else None
        return [x.value for x in v] if v else []

    def dim_l(obj, attr):
        d = getattr(obj, attr, None) if obj else None
        return _meas(d.length if d else None)

    def dim_w(obj, attr):
        d = getattr(obj, attr, None) if obj else None
        return _meas(d.width if d else None)

    def meas(obj, attr):
        return _meas(getattr(obj, attr, None) if obj else None)

    return {
        'family':                           fam,
        'genus':                            gen,
        'species':                          sp,
        'common_name':                      c.common_name if c else None,
        'wagner_pg_number':                 c.wagner_pg_number if c else None,
        'description':                      c.description.value if c and c.description else None,
        'infraspecific_epithet':            c.infraspecific_epithet if c else None,
        'stem_hair_type':                   enums(st, 'stem_hair_type'),
        'phyllotaxy_type':                  enums(lf, 'phyllotaxy_type'),
        'breeding_type':                    enums(lft, 'breeding_type'),
        'inflorescence_type':               enums(inf, 'inflorescence_type'),
        'ray_color':                        cap.ray_color if cap else None,
        'floret_color':                     cap.floret_color if cap else None,
        'spathe_color':                     cap.spathe_color if cap else None,
        'perianth_outer_color':             fl.perianth_outer_color if fl else None,
        'perianth_inner_color':             fl.perianth_inner_color if fl else None,
        'perianth_color':                   fl.perianth_color if fl else None,
        'labellum_color':                   fl.labellum_color if fl else None,
        'corolla_type':                     enums(fl, 'corolla_type'),
        'staminate_corolla_type':           None,
        'pistillate_corolla_type':          None,
        'corolla_color':                    fl.corolla_color if fl else None,
        'fruit_type':                       enums(fr, 'fruit_type'),
        'ploidy':                           rp.ploidy if rp else [],
        'chromosome_number':                rp.chromosome_number if rp else [],
        'average_chromosome_number':        None,
        'origin':                           enums(ds, 'origin'),
        'status':                           enums(ds, 'status'),
        'life_form_type':                   enums(lft, 'life_form_type'),
        'leaf_type':                        enums(lf, 'leaf_type'),
        'leaf_margin_type':                 enums(lf, 'leaf_margin_type'),
        'leaf_shape_type':                  enums(lf, 'leaf_shape_type'),
        'juvenile_leaf_type':               enums(jl, 'juvenile_leaf_type'),
        'juvenile_leaf_margin_type':        enums(jl, 'juvenile_leaf_margin_type'),
        'juvenile_leaf_shape_type':         enums(jl, 'juvenile_leaf_shape_type'),
        'leaflets_leaf_type':               enums(ll, 'leaflets_leaf_type'),
        'leaflets_leaf_margin_type':        enums(ll, 'leaflets_leaf_margin_type'),
        'leaflets_leaf_shape_type':         enums(ll, 'leaflets_leaf_shape_type'),
        'leaf_hair_type':                   enums(li, 'leaf_hair_type'),
        'leaf_hair_upper_type':             enums(li, 'upper_leaf_hair_type'),
        'leaf_hair_lower_type':             enums(li, 'lower_leaf_hair_type'),
        'juvenile_leaf_hair_type':          enums(jl, 'juvenile_leaf_hair_type'),
        'island_type':                      enums(ds, 'island_type'),
        'hawaiian_name':                    set(c.hawaiian_name) if c and c.hawaiian_name else set(),
        'stem_height':                      meas(st, 'stem_height'),
        'leaf_length':                      dim_l(lf, 'leaf_dimensions'),
        'leaf_width':                       dim_w(lf, 'leaf_dimensions'),
        'juvenile_leaf_length':             dim_l(jl, 'juvenile_leaf_dimensions'),
        'juvenile_leaf_width':              dim_w(jl, 'juvenile_leaf_dimensions'),
        'leaflets_leaf_length':             dim_l(ll, 'leaflets_leaf_dimensions'),
        'leaflets_leaf_width':              dim_w(ll, 'leaflets_leaf_dimensions'),
        'petioles':                         meas(lf, 'petiole_length'),
        'staminate_inflorescence_length':   dim_l(inf, 'staminate_inflorescence_dimensions'),
        'staminate_inflorescence_width':    dim_w(inf, 'staminate_inflorescence_dimensions'),
        'pistillate_inflorescence_length':  dim_l(inf, 'pistillate_inflorescence_dimensions'),
        'pistillate_inflorescence_width':   dim_w(inf, 'pistillate_inflorescence_dimensions'),
        'inflorescence_flower_length':      dim_l(inf, 'inflorescence_dimensions'),
        'inflorescence_flower_width':       dim_w(inf, 'inflorescence_dimensions'),
        'flower_length':                    dim_l(fl, 'flower_dimensions'),
        'flower_width':                     dim_w(fl, 'flower_dimensions'),
        'rachis_length':                    meas(inf, 'rachis_length'),
        'rachis_diameter':                  meas(inf, 'rachis_diameter'),
        'head_length':                      meas(cap, 'head_length'),
        'head_diameter':                    meas(cap, 'head_diameter'),
        'bur_length':                       meas(fr, 'bur_length'),
        'tepal_length':                     meas(fl, 'tepal_length'),
        'staminate_tepal_length':           meas(fl, 'staminate_tepal_length'),
        'pistillate_tepal_length':          meas(fl, 'pistillate_tepal_length'),
        'ray_length':                       dim_l(cap, 'ray_dimensions'),
        'ray_width':                        dim_w(cap, 'ray_dimensions'),
        'florets_length':                   meas(cap, 'florets_length'),
        'involucre_length':                 dim_l(inv, 'involucre_dimensions'),
        'involucre_width':                  dim_w(inv, 'involucre_dimensions'),
        'staminate_involucre_length':       meas(inv, 'staminate_involucre_length'),
        'pistilate_involucre_length':       meas(inv, 'pistillate_involucre_length'),
        'bract_length':                     dim_l(br, 'bract_dimensions'),
        'bract_width':                      dim_w(br, 'bract_dimensions'),
        'bract_lower_length':               meas(br, 'bract_lower_length'),
        'bract_outer_length':               meas(br, 'bract_outer_length'),
        'bracteoles_length':                dim_l(br, 'bracteoles_dimensions'),
        'bracteoles_width':                 dim_w(br, 'bracteoles_dimensions'),
        'pedicel_length':                   dim_l(inf, 'pedicel_dimensions'),
        'pedicel_width':                    dim_w(inf, 'pedicel_dimensions'),
        'pistillate_pedicel_length':        dim_l(inf, 'pistillate_pedicel_dimensions'),
        'hypanthium_length':                dim_l(inf, 'hypanthium_dimensions'),
        'hypanthium_width':                 dim_w(inf, 'hypanthium_dimensions'),
        'peduncle_length':                  dim_l(inf, 'peduncle_dimensions'),
        'peduncle_width':                   dim_w(inf, 'peduncle_dimensions'),
        'spathe_length':                    dim_l(cap, 'spathe_dimensions'),
        'spathe_width':                     dim_w(cap, 'spathe_dimensions'),
        'spadix_length':                    meas(cap, 'spadix_length'),
        'perianth_length':                  dim_l(fl, 'perianth_dimensions'),
        'perianth_width':                   dim_w(fl, 'perianth_dimensions'),
        'perianth_outer_length':            dim_l(fl, 'perianth_outer_dimensions'),
        'perianth_outer_width':             dim_w(fl, 'perianth_outer_dimensions'),
        'perianth_inner_length':            dim_l(fl, 'perianth_inner_dimensions'),
        'perianth_inner_width':             dim_w(fl, 'perianth_inner_dimensions'),
        'perianth_tube_length':             meas(fl, 'perianth_tube_length'),
        'perianth_lobes_length':            dim_l(fl, 'perianth_lobes_dimensions'),
        'perianth_lobes_width':             dim_w(fl, 'perianth_lobes_dimensions'),
        'staminate_perianth_tube_length':   meas(fl, 'staminate_perianth_tube_length'),
        'pistillate_perianth_tube_length':  meas(fl, 'pistillate_perianth_tube_length'),
        'pappus_length':                    meas(cap, 'pappus_length'),
        'umbellet_length':                  meas(inf, 'umbellet_length'),
        'labellum_length':                  dim_l(fl, 'labellum_dimensions'),
        'labellum_width':                   dim_w(fl, 'labellum_dimensions'),
        'calyx_length':                     dim_l(fl, 'calyx_dimensions'),
        'calyx_width':                      dim_w(fl, 'calyx_dimensions'),
        'calyx_teeth_length':               dim_l(fl, 'calyx_teeth_dimensions'),
        'calyx_teeth_width':                dim_w(fl, 'calyx_teeth_dimensions'),
        'calyx_lobes_length':               dim_l(fl, 'calyx_lobe_dimensions'),
        'calyx_lobes_width':                dim_w(fl, 'calyx_lobe_dimensions'),
        'upper_calyx_length':               meas(fl, 'upper_calyx_length'),
        'lower_calyx_length':               meas(fl, 'lower_calyx_length'),
        'inner_calyx_lobes_length':         dim_l(fl, 'inner_calyx_lobes_dimensions'),
        'inner_calyx_lobes_width':          dim_w(fl, 'inner_calyx_lobes_dimensions'),
        'outer_calyx_lobes_length':         dim_l(fl, 'outer_calyx_lobes_dimensions'),
        'outer_calyx_lobes_width':          dim_w(fl, 'outer_calyx_lobes_dimensions'),
        'calyx_tube_length':                dim_l(fl, 'calyx_tube_dimensions'),
        'calyx_tube_width':                 dim_w(fl, 'calyx_tube_dimensions'),
        'male_calyx_length':                dim_l(fl, 'male_calyx_dimensions'),
        'male_calyx_width':                 dim_w(fl, 'male_calyx_dimensions'),
        'male_calyx_lobes_length':          dim_l(fl, 'male_calyx_lobe_dimensions'),
        'male_calyx_lobes_width':           dim_w(fl, 'male_calyx_lobe_dimensions'),
        'female_calyx_length':              dim_l(fl, 'female_calyx_dimensions'),
        'female_calyx_width':               dim_w(fl, 'female_calyx_dimensions'),
        'female_calyx_lobes_length':        dim_l(fl, 'female_calyx_lobe_dimensions'),
        'female_calyx_lobes_width':         dim_w(fl, 'female_calyx_lobe_dimensions'),
        'male_calyx_lobes_length_inner':    meas(fl, 'male_calyx_inner_lobe_length'),
        'male_calyx_lobes_length_outer':    dim_l(fl, 'male_calyx_outer_lobe_dimensions'),
        'male_calyx_lobes_width_outer':     dim_w(fl, 'male_calyx_outer_lobe_dimensions'),
        'male_calyx_tube_length':           meas(fl, 'male_calyx_tube_length'),
        'female_calyx_lobes_length_inner':  dim_l(fl, 'female_calyx_inner_lobe_dimensions'),
        'female_calyx_lobes_length_outer':  dim_l(fl, 'female_calyx_outer_lobe_dimensions'),
        'female_calyx_lobes_width_inner':   dim_w(fl, 'female_calyx_inner_lobe_dimensions'),
        'female_calyx_lobes_width_outer':   dim_w(fl, 'female_calyx_outer_lobe_dimensions'),
        'inner_calyx_length':               meas(fl, 'inner_calyx_length'),
        'outer_calyx_length':               meas(fl, 'outer_calyx_length'),
        'corolla_length':                   dim_l(fl, 'corolla_dimensions'),
        'corolla_width':                    dim_w(fl, 'corolla_dimensions'),
        'corolla_tube_length':              dim_l(fl, 'corolla_tube_dimensions'),
        'corolla_tube_width':               dim_w(fl, 'corolla_tube_dimensions'),
        'corolla_lobes_length':             dim_l(fl, 'corolla_lobe_dimensions'),
        'corolla_lobes_width':              dim_w(fl, 'corolla_lobe_dimensions'),
        'upper_corolla_length':             meas(fl, 'upper_corolla_length'),
        'lower_corolla_length':             meas(fl, 'lower_corolla_length'),
        'upper_corolla_lobes_length':       meas(fl, 'upper_corolla_lobes_length'),
        'lower_corolla_lobes_length':       meas(fl, 'lower_corolla_lobes_length'),
        'corolla_lip_length':               meas(fl, 'corolla_lip_length'),
        'staminate_corolla_length':         meas(fl, 'staminate_corolla_length'),
        'pistillate_corolla_length':        meas(fl, 'pistillate_corolla_length'),
        'staminate_corolla_tube_length':    dim_l(fl, 'staminate_corolla_tube_dimensions'),
        'staminate_corolla_tube_width':     dim_w(fl, 'staminate_corolla_tube_dimensions'),
        'pistillate_corolla_tube_length':   dim_l(fl, 'pistillate_corolla_tube_dimensions'),
        'pistillate_corolla_tube_width':    dim_w(fl, 'pistillate_corolla_tube_dimensions'),
        'female_corolla_lobes_length':      dim_l(fl, 'female_corolla_lobe_dimensions'),
        'female_corolla_lobes_width':       dim_w(fl, 'female_corolla_lobe_dimensions'),
        'male_corolla_lobes_length':        dim_l(fl, 'male_corolla_lobe_dimensions'),
        'male_corolla_lobes_width':         dim_w(fl, 'male_corolla_lobe_dimensions'),
        'fruit_length':                     meas(fr, 'fruit_length'),
        'fruit_width':                      meas(fr, 'fruit_width'),
        'fruit_diameter':                   meas(fr, 'fruit_diameter'),
        'seeds_perfruit':                   meas(fr, 'seeds_perfruit'),
        'seed_length':                      meas(fr, 'seed_length'),
        'seed_width':                       meas(fr, 'seed_width'),
        'seed_diameter':                    meas(fr, 'seed_diameter'),
        'pistillate_peduncle_length':       dim_l(inf, 'pistillate_peduncle_dimensions'),
        'pistillate_peduncle_width':        dim_w(inf, 'pistillate_peduncle_dimensions'),
        'staminate_pedicel_length':         dim_l(inf, 'staminate_pedicel_dimensions'),
        'staminate_peduncle_length':        dim_l(inf, 'staminate_peduncle_dimensions'),
        'staminate_peduncle_width':         dim_w(inf, 'staminate_peduncle_dimensions'),
        'species_key': f"{fam.lower()}_{gen.lower()}_{sp.lower()}" if fam and gen and sp else None,
    }

def results_to_df(result):
    row = result_to_row(result)
    gs_cols = manual_extracted_data_df.columns.tolist()
    return pd.DataFrame([row])[gs_cols]


In [34]:
results_to_df(res)

,family,genus,species,common_name,wagner_pg_number,description,infraspecific_epithet,stem_hair_type,phyllotaxy_type,breeding_type,inflorescence_type,ray_color,floret_color,spathe_color,perianth_outer_color,perianth_inner_color,perianth_color,labellum_color,corolla_type,staminate_corolla_type,pistillate_corolla_type,corolla_color,fruit_type,ploidy,chromosome_number,average_chromosome_number,origin,status,life_form_type,leaf_type,leaf_margin_type,leaf_shape_type,juvenile_leaf_type,juvenile_leaf_margin_type,juvenile_leaf_shape_type,leaflets_leaf_type,leaflets_leaf_margin_type,leaflets_leaf_shape_type,leaf_hair_type,leaf_hair_upper_type,leaf_hair_lower_type,juvenile_leaf_hair_type,island_type,hawaiian_name,stem_height,leaf_length,leaf_width,juvenile_leaf_length,juvenile_leaf_width,leaflets_leaf_length,leaflets_leaf_width,petioles,staminate_inflorescence_length,staminate_inflorescence_width,pistillate_inflorescence_length,pistillate_inflorescence_width,inflorescence_flower_length,inflorescence_flower_width,flower_length,flower_width,rachis_length,rachis_diameter,head_length,head_diameter,bur_length,tepal_length,staminate_tepal_length,pistillate_tepal_length,ray_length,ray_width,florets_length,involucre_length,involucre_width,staminate_involucre_length,pistilate_involucre_length,bract_length,bract_width,bract_lower_length,bract_outer_length,bracteoles_length,bracteoles_width,pedicel_length,pedicel_width,pistillate_pedicel_length,hypanthium_length,hypanthium_width,peduncle_length,peduncle_width,spathe_length,spathe_width,spadix_length,perianth_length,perianth_width,perianth_outer_length,perianth_outer_width,perianth_inner_length,perianth_inner_width,perianth_tube_length,perianth_lobes_length,perianth_lobes_width,staminate_perianth_tube_length,pistillate_perianth_tube_length,pappus_length,umbellet_length,labellum_length,labellum_width,calyx_length,calyx_width,calyx_teeth_length,calyx_teeth_width,calyx_lobes_length,calyx_lobes_width,upper_calyx_length,lower_calyx_length,inner_calyx_lobes_length,inner_calyx_lobes_width,outer_calyx_lobes_length,outer_calyx_lobes_width,calyx_tube_length,calyx_tube_width,male_calyx_length,male_calyx_width,male_calyx_lobes_length,male_calyx_lobes_width,female_calyx_length,female_calyx_width,female_calyx_lobes_length,female_calyx_lobes_width,male_calyx_lobes_length_inner,male_calyx_lobes_length_outer,male_calyx_lobes_width_outer,male_calyx_tube_length,female_calyx_lobes_length_inner,female_calyx_lobes_length_outer,female_calyx_lobes_width_inner,female_calyx_lobes_width_outer,inner_calyx_length,outer_calyx_length,corolla_length,corolla_width,corolla_tube_length,corolla_tube_width,corolla_lobes_length,corolla_lobes_width,upper_corolla_length,lower_corolla_length,upper_corolla_lobes_length,lower_corolla_lobes_length,corolla_lip_length,staminate_corolla_length,pistillate_corolla_length,staminate_corolla_tube_length,staminate_corolla_tube_width,pistillate_corolla_tube_length,pistillate_corolla_tube_width,female_corolla_lobes_length,female_corolla_lobes_width,male_corolla_lobes_length,male_corolla_lobes_width,fruit_length,fruit_width,fruit_diameter,seeds_perfruit,seed_length,seed_width,seed_diameter,pistillate_peduncle_length,pistillate_peduncle_width,staminate_pedicel_length,staminate_peduncle_length,staminate_peduncle_width,species_key
0,Apiaceae,Coriandrum,sativum,"Coriander, Chinese parsley",0,Dicots,None,[],[],[],[UMBEL],None,None,None,None,None,None,None,[],None,None,None,[],[2n],"[22, 23, 24, 25, 26, 28, 30]",None,[NATURALIZED],[NATURALIZED],[],[COMPOUND],[],"[OVATE, FLABELLATE, CUNEATE]",[],[],[],[COMPOUND],[TEETH],"[FLABELLATE, CUNEATE]",[],[],[],[],[OAHU],{},"{'exmin': None, 'min': 2.0, 'max': 7.0, 'exmax...","{'exmin': None, 'min': 3.0, 'max': 15.0, 'exma...","{'exmin': None, 'min': 2.0, 'max': 10.0, 'exma...",None,None,"{'exmin': None, 'min': 1.0, 'max': 2.0, 'exmax...","{'exmin': None, 'min': 0.5, 'max': 1.0, 'exmax...","{'exmin': None, 'min': 1.0, 'max': 15.0, 'exma...",None,None,None,None,None,None,

In [59]:
res, flags = await extract_plant(text_inputs['Coriandrum_sativum'.upper()], model)

Attempt 1/2: correcting ['leaf_morphology', 'leaflet_morphology', 'distribution']
  leaf_morphology.leaf_margin_type: Source states basal leaves are 'ternately or pinnately lobed to pinnately compound' and leaflets are 'toothed or incised'; assigning 'TEETH' at the whole-leaf level is not specific to the leaf field and derives from leaflet margins.
  leaflet_morphology.leaflets_leaf_type: Leaflets are parts of a compound leaf; the source does not describe leaflets themselves as 'COMPOUND'. This value is not specific to the leaflet field.
  distribution.origin: Source says the species is 'Native to the Mediterranean region; in Hawai‘i cultivated and naturalized on O‘ahu.' Marking origin as 'NATURALIZED' is not specific/correct for origin; naturalized describes status in Hawai‘i, not geographic origin.
Attempt 2/2: correcting ['leaf_morphology', 'leaflet_morphology', 'distribution']
  leaf_morphology.leaf_shape_type: Includes CUNEATE and FLABELLATE, which in the source text describe the 

In [60]:
for i in flags:
    print(i)

('has_core', Flag(value=True, description='True — the passage includes core taxonomic/identification metadata: "Family: APIACEAE," "Genus: CORIANDRUM," "Species: CORIANDRUM SATIVUM," plus identifying description such as "Plants 2–7 dm tall... Fruit 1.5–5 mm in diameter."'))
('has_life_form', Flag(value=False, description='Not mentioned. The passage gives size and leaf/fruit characters (e.g., "Plants 2–7 dm tall") but does not explicitly state a life form such as annual herb, shrub, tree, or vine.'))
('has_outer_flower_morphology', Flag(value=True, description='True — the passage describes outer flower morphology: "calyx teeth ovate-lanceolate, 0.5–0.8 mm long."'))
('has_fruit_morphology', Flag(value=True, description='True — the passage describes fruit morphology: “Fruit 1.5–5 mm in diameter.”'))
('has_inflorescence_specific_morphology', Flag(value=True, description='"Peduncles 3–10 cm long or occasionally abortive, rays 2–8, 1–2.5 cm long" — the passage clearly mentions ray morphology

In [61]:
res

{'core': CoreFieldsModel(family='Apiaceae', genus='Coriandrum', species='sativum', infraspecific_epithet=None, common_name='Coriander, Chinese parsley', hawaiian_name=None, wagner_pg_number='0', description=<Description.DICOTS: 'Dicots'>),
 'outer_flower_morphology': OuterFlowerMorphologyModel(corolla_type=None, corolla_color=None, perianth_color=None, perianth_dimensions=None, corolla_dimensions=None, calyx_dimensions=None, labellum_color=None, labellum_dimensions=None, calyx_teeth_dimensions=Dimensions(length=Measurements(min=0.5, max=0.8, extreme_min=None, extreme_max=None, unit='mm'), width=None), calyx_lobe_dimensions=None, upper_calyx_length=None, lower_calyx_length=None, inner_calyx_length=None, outer_calyx_length=None, inner_calyx_lobes_dimensions=None, outer_calyx_lobes_dimensions=None, male_calyx_dimensions=None, female_calyx_dimensions=None, male_calyx_lobe_dimensions=None, male_calyx_inner_lobe_length=None, male_calyx_outer_lobe_dimensions=None, female_calyx_lobe_dimensions

In [62]:
dfs = []

for i in text_inputs.keys():
    print(f"Processing {i}...")
    res, flags = await extract_plant(text_inputs[i], model)
    
    species_key = f"{res['core'].family}_{res['core'].genus}_{res['core'].species}".lower()
    # print(species_key.lower(), "\n")
    man_row = manual_extracted_data_df[manual_extracted_data_df['species_key'] == species_key]
    auto_df = results_to_df(res)
        
    dfs.append(auto_df)
    dfs.append(man_row)
    # break

# Combine all results and output to CSV
if dfs:
    final_df = pd.concat(dfs, ignore_index=True)
    final_df.to_csv("extracted_test_3.csv", index=False)
    print("Saved to extracted_test_2.csv")

Processing DAUCUS_PUSILLUS...
Attempt 1/2: correcting ['Inflorescence_Specific', 'bract_involucre_morphology', 'distribution']
  Inflorescence_Specific.floret_color: Source says flowers are white, but does not describe florets; 'floret_color' is not specific to this inflorescence/species description.
  bract_involucre_morphology.bracteoles_dimensions: The extracted 2–9 mm length comes from pedicels ('pedicels unequal, 2–9 mm long'), while source only says involucel bractlets are about equalling the pedicels and gives no direct bractlet measurement.
  distribution.status: Source says 'introduced or perhaps native' and discusses possible native/pre-contact introduction status; 'NATURALIZED' is not supported and contradicts that uncertainty.
All groups valid after 1 correction(s).
Processing HYDROCOTYLE_BOWLESIOIDES...
All groups valid.
Processing NESTEGIS_SANDWICENSIS...
Attempt 1/2: correcting ['leaf_morphology']
  leaf_morphology.phyllotaxy_type: Source text states flowers are in axill

In [ ]:
res

In [ ]:
sample = text_inputs['OENOTHERA_STRICTA'.upper()]
res, flags = await extract_plant(sample, model)
print(res)

In [ ]:
row = result_to_row(res)
species_key = row['species_key']
man_row = manual_extracted_data_df[manual_extracted_data_df['species_key'] == species_key]
print(f"Species: {species_key}\n")

for col_name, auto_value in row.items():
    col = man_row.get(col_name)
    if col is None or col.empty:
        continue
    manual_value = col.values[0]
    print(f"{col_name}:")
    print(f"  Auto:   {auto_value}")
    print(f"  Manual: {manual_value}")
    print()


In [ ]:
sample

In [ ]:
sample

In [ ]:
final_df = pd.concat(dfs, ignore_index=True)
final_df

In [ ]:
pwd


In [ ]:
final_df.to_csv("extracted_plants.csv", index=False)

In [ ]:
sample = text_inputs['DAUCUS_PUSILLUS'.upper()]
sample

In [ ]:
res

In [ ]:
sample

In [ ]:
final_df['species_key']

In [ ]:
class AreAnntoationsEqual(BaseModel):
    are_equal: bool = Field(..., description="Are the two values synonymous?")
    # justification: str = Field(..., description="Justification of the propose value for are_equal. Justifications should be as concise as possible.")
    

## Setting the prompt and model for validation agent
validation_agent = Agent(
    # model="openai:o3-mini",
    model="openai:gpt-5.4-mini",
    output_type=AreAnntoationsEqual,
    system_prompt = """You are an expert taxonomist. You are comparing the outcome of a manually extracted result versus an automatically extracted result. You need to compare the automatic results and determine whether the result is synonymous or equal the manual one; taking into consideration
    linguisitc and formatting nuances. If the measurements are correct but are seemigly in the wrong units, you can mark that as the results being the same = True. Your answer is whether the two results are similar True/False.""",
)